### Things to do

- add support for nz>2
- add the ZB20 analytic (?) closure in too
- figure out how to include different steps in a curriculum if the models are trained on different sizes
- fix GAN
- add 95% bounds for error for trajectory rollout loss
- check and then maybe togglise the learning rate cosine scheduler
- combined losses aren't normalised so larger loss scales dominate balance 
- data efficiency plots perhaps
- make a better, more robust system for timing checks
- run some weak/strong jet regime checks
- add Rossby wavelength to the KE plot
-  CNNs haveeight hidden convolutional layers with square kernels, first two layers with size 5 and in circular padding mode,and the remaining six layers have size 3 in constant padding mode. The stride for the cross‐correlation is 2 for thefirst two layers, and 1 for other layers. ReLU is chosen as the activation function between each layer. No batchnormalization is performed; (ross has a hard constraint, yan doesnt use it !)

In [3]:

import os
import sys
import time
import numpy as np
import scipy
import jax
import jax.numpy as jnp
import equinox as eqx

print("Starting native library tests...", flush=True)

print("\n1. NumPy")
x = np.random.randn(2000, 2000)
y = x @ x.T
print("NumPy OK:", y.shape, flush=True)

print("\n2. SciPy")
from scipy.linalg import eigh
eigh(np.eye(100))
print("SciPy OK", flush=True)

print("\n3. JAX")
x = jnp.ones((2000, 2000))
y = x @ x.T
y.block_until_ready()
print("JAX OK:", y.shape, flush=True)

print("\n4. Equinox")
model = eqx.nn.MLP(
    in_size=10,
    out_size=2,
    width_size=32,
    depth=2,
    key=jax.random.PRNGKey(42)
)
print("Equinox OK:", model, flush=True)

print("\nALL TESTS COMPLETED")


Starting native library tests...

1. NumPy
NumPy OK: (2000, 2000)

2. SciPy
SciPy OK

3. JAX
JAX OK: (2000, 2000)

4. Equinox
Equinox OK: MLP(
  layers=(
    Linear(
      weight=f32[32,10],
      bias=f32[32],
      in_features=10,
      out_features=32,
      use_bias=True
    ),
    Linear(
      weight=f32[32,32],
      bias=f32[32],
      in_features=32,
      out_features=32,
      use_bias=True
    ),
    Linear(
      weight=f32[2,32],
      bias=f32[2],
      in_features=32,
      out_features=2,
      use_bias=True
    )
  ),
  activation=<PjitFunction of <function relu at 0x000001967D9B20C0>>,
  final_activation=<function <lambda>>,
  use_bias=True,
  use_final_bias=True,
  in_size=10,
  out_size=2,
  width_size=32,
  depth=2
)

ALL TESTS COMPLETED


In [2]:

import os
import sys
import platform
import importlib.metadata as metadata
from pathlib import Path

print("=" * 65)
print("PYTHON ENVIRONMENT DIAGNOSTIC")
print("=" * 65)

print("\n[1] PYTHON")
print("Executable:", sys.executable)
print("Prefix:", sys.prefix)
print("Version:", sys.version)
print("Platform:", platform.platform())

print("\n[2] CONDA VARIABLES")
for key in [
    "CONDA_PREFIX",
    "CONDA_DEFAULT_ENV",
    "CONDA_SHLVL",
    "CONDA_EXE",
    "PYTHONPATH",
    "PYTHONHOME",
    "CUDA_PATH",
    "CUDA_HOME",
    "CUDA_VISIBLE_DEVICES",
]:
    print(f"{key}: {os.environ.get(key, '<not set>')}")

print("\n[3] EXPECTED CONDA DIRECTORIES")
prefix = Path(sys.prefix)

for p in [
    prefix,
    prefix / "Library" / "bin",
    prefix / "Library" / "mingw-w64" / "bin",
    prefix / "Library" / "usr" / "bin",
    prefix / "Scripts",
    prefix / "bin",
]:
    print(f"{'EXISTS' if p.exists() else 'MISSING'}: {p}")

print("\n[4] PATH ORDER")
for i, p in enumerate(os.environ.get("PATH", "").split(os.pathsep)):
    print(f"{i:02d}: {p}")

print("\n[5] IMPORTANT PACKAGE VERSIONS")
for package in [
    "numpy",
    "scipy",
    "torch",
    "jax",
    "jaxlib",
    "equinox",
    "ipykernel",
    "jupyter",
    "tensorflow",
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\n[6] PYTHON MODULE SEARCH PATH")
for p in sys.path:
    print(p)

print("\n" + "=" * 65)
print("DIAGNOSTIC COMPLETE")
print("=" * 65)


import jax

print("JAX version:", jax.__version__)
print("Default backend:", jax.default_backend())
print("Devices:", jax.devices())

print("\nBackend platforms:")
for device in jax.devices():
    print(device.platform, device.device_kind)



PYTHON ENVIRONMENT DIAGNOSTIC

[1] PYTHON
Executable: c:\Users\C5044892\AppData\Local\anaconda3\envs\QG\python.exe
Prefix: c:\Users\C5044892\AppData\Local\anaconda3\envs\QG
Version: 3.13.0 | packaged by conda-forge | (main, Nov 27 2024, 19:03:49) [MSC v.1942 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0

[2] CONDA VARIABLES
CONDA_PREFIX: C:\Users\C5044892\AppData\Local\anaconda3\envs\QG
CONDA_DEFAULT_ENV: QG
CONDA_SHLVL: 1
CONDA_EXE: C:\Users\C5044892\AppData\Local\miniconda3\Scripts\conda.exe
PYTHONPATH: <not set>
PYTHONHOME: <not set>
CUDA_PATH: <not set>
CUDA_HOME: <not set>
CUDA_VISIBLE_DEVICES: <not set>

[3] EXPECTED CONDA DIRECTORIES
EXISTS: c:\Users\C5044892\AppData\Local\anaconda3\envs\QG
EXISTS: c:\Users\C5044892\AppData\Local\anaconda3\envs\QG\Library\bin
MISSING: c:\Users\C5044892\AppData\Local\anaconda3\envs\QG\Library\mingw-w64\bin
MISSING: c:\Users\C5044892\AppData\Local\anaconda3\envs\QG\Library\usr\bin
EXISTS: c:\Users\C5044892\AppData\Local\anaconda3\envs\QG\Scr

In [2]:
import os
import sys
from pathlib import Path

prefix = Path(sys.prefix)

conda_paths = [prefix, prefix / "Library" / "bin", prefix / "Scripts"]

existing = os.environ["PATH"].split(os.pathsep)
new_paths = [str(p) for p in conda_paths]

for p in existing:
    if p and p.lower() not in [x.lower() for x in new_paths]:
        new_paths.append(p)

os.environ["PATH"] = os.pathsep.join(new_paths)

In [2]:
%run ../run.py 

2026-11-21 17:22:27,364 INFO model.ML.utils.loss: Using combined loss functions: ['mse']
2026-11-21 17:22:27,366 INFO __main__: Requested device: cpu, using device: CPU
2026-11-21 17:25:46,248 INFO __main__: Fine timestep is 2e+03s and coarsened timestep is 3.2e+04s. Model spinup for 20.00 eddy turnover times (~1.9e+04 high-res steps). Training horizon is 2000 low-res steps (~740.74 days). Validation plotting window is 50 steps (~18.52 days).


GroupNotFoundError: No group found in store 'c:\\Users\\C5044892\\Documents\\Code\\QG\\data\\data_hr512_nx32_13\\trajectories.zarr' at path ''